# AI Personal Stylist & Shopping Agent

**Hackathon Demo — AMD Instinct MI300X (192GB VRAM)**

An end-to-end agentic shopping assistant that:
- Searches Amazon, Flipkart, Myntra, Ajio, Nykaa via DuckDuckGo
- Analyzes ratings & reviews with local LLMs
- Generates complete occasion-based outfits
- Analyzes user photos for personalized styling
- Ranks products by rating, review quality, budget fit & style relevance

### Architecture

```
User → Orchestrator → [Vision Tool] → Stylist Tool → Product Search → Review Analysis → Ranking → Response
```

| Component | Model | Endpoint |
|-----------|-------|----------|
| Text (reasoning, styling, reviews) | `Qwen/Qwen3-32B` | `http://localhost:8000/v1` |
| Vision (photo analysis) | `Qwen/Qwen2.5-VL-7B-Instruct` | `http://localhost:8001/v1` |

> **Note:** Start both vLLM servers before running agent cells (see Section 1).

### Implementation Plan

| Step | Module | Key Functions |
|------|--------|---------------|
| 1 | Environment Setup | pip install, ROCm GPU check |
| 2 | vLLM Clients | `text_client`, `vision_client`, `llm_json_call()` |
| 3 | Data Models | `UserProfile`, `Outfit`, `Product`, `ReviewSummary`, `Recommendation`, `OutfitRecommendation` |
| 4 | Vision Tool | `analyze_user_image()` → Qwen2.5-VL |
| 5 | Stylist Tool | `generate_outfits()` → Qwen3-32B |
| 6 | Product Search | `search_products()` → DuckDuckGo + site: filters |
| 7 | Review Analysis | `analyze_reviews()` → Qwen3-32B |
| 8 | Ranking | `rank_products()` → weighted scoring |
| 9 | Shopping Agent | `recommend_products()` → search + review + rank |
| 10 | Outfit Agent | `build_complete_outfits()` → stylist + search + assemble |
| 11 | Orchestrator | `StylistOrchestrator.process()` → intent routing |
| 12 | Gradio UI | Chat + image upload + HTML product/outfit cards |
| 13 | Examples | 3 demo scenarios |

## Section 1 — Environment Setup

Install dependencies and verify ROCm GPU availability on MI300X.

> **Important:** Qwen3 has **no** separate `-Instruct` repo on Hugging Face.
> Use **`Qwen/Qwen3-32B`** (already instruction-tuned).

**Check how many GPUs you have first (run in terminal):**
```bash
rocm-smi --showproductname
# or: python -c "import torch; print(torch.cuda.device_count())"
```

### Option A — Two GPUs available (use GPU 0 + GPU 1)

```bash
# Terminal 1 — Text model on GPU 0
HIP_VISIBLE_DEVICES=0 python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen3-32B \
  --host 0.0.0.0 --port 8000 \
  --dtype bfloat16 \
  --max-model-len 32768 \
  --gpu-memory-utilization 0.85 \
  --trust-remote-code

# Terminal 2 — Vision model on GPU 1
HIP_VISIBLE_DEVICES=1 python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen2.5-VL-7B-Instruct \
  --host 0.0.0.0 --port 8001 \
  --dtype bfloat16 \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.30 \
  --limit-mm-per-prompt.image 4 \
  --trust-remote-code
```

### Option B — Single GPU (typical hackathon Jupyter pod) — **use this if you see `No HIP GPUs are available`**

`HIP_VISIBLE_DEVICES=1` fails when only **one** GPU exists (index 0). Run **both servers on GPU 0** with lower memory splits:

```bash
# Terminal 1 — Text model (start first, wait until "Application startup complete")
HIP_VISIBLE_DEVICES=0 python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen3-32B \
  --host 0.0.0.0 --port 8000 \
  --dtype bfloat16 \
  --max-model-len 16384 \
  --gpu-memory-utilization 0.55 \
  --trust-remote-code

# Terminal 2 — Vision model (same GPU 0, NOT gpu 1)
HIP_VISIBLE_DEVICES=0 python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen2.5-VL-7B-Instruct \
  --host 0.0.0.0 --port 8001 \
  --dtype bfloat16 \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.18 \
  --limit-mm-per-prompt.image 4 \
  --trust-remote-code
```

> **Why the error?** `HIP_VISIBLE_DEVICES=1` hides GPU 0 and exposes only physical GPU #1.
> On a 1-GPU machine there is no GPU #1 → `RuntimeError: No HIP GPUs are available`.

**First launch downloads ~65GB (text) + ~15GB (vision) from Hugging Face.**

In [ ]:
%%bash
# Install all required packages (run once)
pip install -q \
  "vllm>=0.6.0" \
  "transformers>=4.45.0" \
  "openai>=1.40.0" \
  "gradio>=4.44.0" \
  "pydantic>=2.7.0" \
  "beautifulsoup4>=4.12.0" \
  "requests>=2.31.0" \
  "ddgs>=9.0.0" \
  "ddgs>=9.0.0" \
  "duckduckgo-search>=6.2.0" \
  "Pillow>=10.0.0" \
  "httpx>=0.27.0" \
  "httpcore>=1.0.9" \
  "tenacity>=8.3.0"

echo "✓ Packages installed"

In [ ]:
import logging
import os
import subprocess
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("stylist_agent")

# ── GPU / ROCm verification ──────────────────────────────────────────
def check_rocm_gpu() -> dict[str, str | int | None]:
    """Verify AMD GPU visibility via ROCm."""
    info: dict[str, str | int | None] = {"platform": "unknown", "gpu_count": 0}
    try:
        result = subprocess.run(
            ["rocm-smi", "--showproductname"],
            capture_output=True, text=True, timeout=10,
        )
        if result.returncode == 0:
            info["platform"] = "ROCm"
            info["gpu_count"] = result.stdout.count("GPU[")
            info["details"] = result.stdout.strip()[:500]
            logger.info("ROCm GPUs detected: %s", info["gpu_count"])
        else:
            logger.warning("rocm-smi unavailable — running in CPU/fallback mode")
    except FileNotFoundError:
        logger.warning("rocm-smi not found — ensure ROCm drivers are installed")
    except subprocess.TimeoutExpired:
        logger.warning("rocm-smi timed out")
    return info

gpu_info = check_rocm_gpu()
gpu_count = int(gpu_info.get("gpu_count") or 0)
print(f"Platform: {gpu_info.get('platform')} | GPUs: {gpu_count}")
if gpu_info.get("details"):
    print(gpu_info["details"][:300])

print("\n── vLLM launch recommendation ──")
if gpu_count >= 2:
    print("✓ 2+ GPUs detected → use Option A in Section 1")
    print("  Text:  HIP_VISIBLE_DEVICES=0  port 8000")
    print("  Vision: HIP_VISIBLE_DEVICES=1  port 8001")
elif gpu_count == 1:
    print("✓ 1 GPU detected → use Option B in Section 1 (both on GPU 0)")
    print("  Text:  HIP_VISIBLE_DEVICES=0  port 8000  gpu-memory-utilization 0.55")
    print("  Vision: HIP_VISIBLE_DEVICES=0  port 8001  gpu-memory-utilization 0.18")
    print("  ⚠ Do NOT use HIP_VISIBLE_DEVICES=1 — causes 'No HIP GPUs are available'")
else:
    print("✗ No GPUs detected — check ROCm driver / container GPU passthrough")

## Section 2 — vLLM Client Setup

Reusable OpenAI-compatible clients for local vLLM endpoints on MI300X.

In [ ]:
import base64
import json
import re
from io import BytesIO
from pathlib import Path
from typing import Any, Literal, TypeVar

import httpx
from openai import OpenAI
from PIL import Image
from pydantic import BaseModel, Field, ValidationError
from tenacity import retry, stop_after_attempt, wait_exponential

# ── Endpoint configuration ───────────────────────────────────────────
TEXT_BASE_URL = os.getenv("TEXT_LLM_URL", "http://localhost:8000/v1")
VISION_BASE_URL = os.getenv("VISION_LLM_URL", "http://localhost:8001/v1")
TEXT_MODEL = os.getenv("TEXT_MODEL", "Qwen/Qwen3-32B")
VISION_MODEL = os.getenv("VISION_MODEL", "Qwen/Qwen2.5-VL-7B-Instruct")

# Qwen3 defaults to thinking mode — disable for faster structured JSON responses
TEXT_CHAT_KWARGS: dict[str, Any] = {
    "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
}

T = TypeVar("T", bound=BaseModel)


def _make_client(base_url: str) -> OpenAI:
    """Create an OpenAI-compatible client for a local vLLM server."""
    return OpenAI(
        base_url=base_url,
        api_key="not-needed",  # vLLM ignores this
        timeout=httpx.Timeout(120.0, connect=10.0),
        max_retries=2,
    )


text_client = _make_client(TEXT_BASE_URL)
vision_client = _make_client(VISION_BASE_URL)


@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=8))
def check_server_health(client: OpenAI, model: str) -> bool:
    """Ping a vLLM server to confirm it is reachable."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "ping"}],
        max_tokens=5,
        temperature=0.0,
        **(TEXT_CHAT_KWARGS if client is text_client else {}),
    )
    return bool(response.choices[0].message.content)


def extract_json_from_text(text: str) -> dict[str, Any]:
    """Extract the first JSON object/array from LLM output."""
    # Strip Qwen3 thinking blocks if present
    if "</redacted_thinking>" in text:
        text = text.split("</redacted_thinking>", 1)[-1].strip()
    brace = text.find("{")
    bracket = text.find("[")
    starts = [i for i in (brace, bracket) if i >= 0]
    if starts:
        text = text[min(starts):]
    # Try fenced code block first
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        text = fence.group(1).strip()
    # Find JSON object or array
    for pattern in (r"\{[\s\S]*\}", r"\[[\s\S]*\]"):
        match = re.search(pattern, text)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                continue
    raise ValueError(f"No valid JSON found in LLM response: {text[:300]}...")


def llm_json_call(
    client: OpenAI,
    model: str,
    messages: list[dict[str, Any]],
    schema_model: type[T],
    temperature: float = 0.3,
    max_tokens: int = 4096,
) -> T:
    """Call LLM and parse response into a Pydantic model."""
    kwargs = TEXT_CHAT_KWARGS if client is text_client else {}
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        **kwargs,
    )
    raw = response.choices[0].message.content or ""
    logger.debug("LLM raw response (first 500 chars): %s", raw[:500])
    try:
        data = extract_json_from_text(raw)
        # Coerce JSON nulls before Pydantic validation
        if isinstance(data, dict):
            data = {k: ("" if v is None and k in ("query", "occasion", "reasoning", "gender") else 0.0 if v is None and k == "budget" else v) for k, v in data.items()}
        return schema_model.model_validate(data)
    except (ValueError, ValidationError) as exc:
        logger.error("JSON parse/validation failed: %s", exc)
        raise


def image_to_base64(image: Image.Image, fmt: str = "JPEG") -> str:
    """Convert PIL Image to base64 data URI."""
    buffer = BytesIO()
    if image.mode not in ("RGB", "L"):
        image = image.convert("RGB")
    image.save(buffer, format=fmt)
    b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
    mime = "image/jpeg" if fmt.upper() == "JPEG" else f"image/{fmt.lower()}"
    return f"data:{mime};base64,{b64}"


# Quick health check (skip if servers not running yet)
for name, client, model in [
    ("Text", text_client, TEXT_MODEL),
    ("Vision", vision_client, VISION_MODEL),
]:
    try:
        check_server_health(client, model)
        print(f"✓ {name} server OK @ {client.base_url}")
    except Exception as exc:
        print(f"✗ {name} server unreachable: {exc}")
        print(f"  → Start vLLM server for {model} before running agent cells")

## Section 3 — Pydantic Data Models

Typed schemas for the entire agent pipeline.

In [ ]:
from enum import Enum


class Gender(str, Enum):
    MALE = "male"
    FEMALE = "female"
    UNISEX = "unisex"
    OTHER = "other"


class Occasion(str, Enum):
    CASUAL = "casual"
    FORMAL = "formal"
    WEDDING = "wedding"
    PARTY = "party"
    OFFICE = "office"
    SPORTS = "sports"
    DATE = "date"
    FESTIVAL = "festival"
    OTHER = "other"


class UserProfile(BaseModel):
    """Derived from vision analysis or user-provided context."""
    skin_tone: str = Field(description="e.g. warm, cool, neutral, deep, fair")
    body_type: str = Field(description="e.g. athletic, slim, curvy, broad, petite")
    style: str = Field(description="Current detected style preference")
    confidence: float = Field(ge=0.0, le=1.0, default=0.7)
    gender: Gender = Gender.UNISEX
    city: str = "Mumbai"
    color_preferences: list[str] = Field(default_factory=list)
    notes: str = ""


class OutfitItem(BaseModel):
    """A single clothing/accessory slot in an outfit."""
    category: Literal["top", "bottom", "footwear", "accessories", "outerwear"]
    description: str = Field(description="Detailed item description for product search")
    color: str = ""
    material: str = ""
    estimated_price_inr: float = Field(ge=0, description="Target price in INR")


class Outfit(BaseModel):
    """A complete outfit concept generated by the stylist."""
    name: str
    occasion: str
    items: list[OutfitItem]
    style_reasoning: str
    total_estimated_budget_inr: float = 0.0

    def model_post_init(self, __context: Any) -> None:
        if self.total_estimated_budget_inr == 0.0 and self.items:
            self.total_estimated_budget_inr = sum(
                i.estimated_price_inr for i in self.items
            )


class Product(BaseModel):
    """A discovered product from e-commerce search."""
    title: str
    price_inr: float = Field(ge=0, default=0.0)
    currency: str = "INR"
    rating: float = Field(ge=0.0, le=5.0, default=0.0)
    review_count: int = Field(ge=0, default=0)
    url: str
    image_url: str = ""
    source: str = Field(description="amazon, flipkart, myntra, ajio, nykaa")
    snippet: str = ""
    raw_reviews: list[str] = Field(default_factory=list)


class ReviewSummary(BaseModel):
    """Summarized review analysis for a product."""
    pros: list[str] = Field(default_factory=list)
    cons: list[str] = Field(default_factory=list)
    overall_sentiment: Literal["positive", "neutral", "negative", "mixed"] = "neutral"
    summary: str = ""
    review_quality_score: float = Field(
        ge=0.0, le=1.0, default=0.5,
        description="0-1 score based on review depth and consistency",
    )


class RankedProduct(BaseModel):
    """Product with computed ranking score."""
    product: Product
    review_summary: ReviewSummary | None = None
    ranking_score: float = Field(ge=0.0, le=100.0, default=0.0)
    score_breakdown: dict[str, float] = Field(default_factory=dict)


class Recommendation(BaseModel):
    """Top product recommendation with explanation."""
    ranked_product: RankedProduct
    rank: int = Field(ge=1)
    explanation: str = Field(description="Why this product was recommended")


class OutfitProductSlot(BaseModel):
    """A filled outfit slot with a real purchasable product."""
    category: str
    product: Product
    review_summary: ReviewSummary | None = None
    ranking_score: float = 0.0


class OutfitRecommendation(BaseModel):
    """Complete purchasable outfit board."""
    outfit: Outfit
    slots: list[OutfitProductSlot]
    total_price_inr: float = 0.0
    stylist_notes: str = ""

    def model_post_init(self, __context: Any) -> None:
        if self.total_price_inr == 0.0 and self.slots:
            self.total_price_inr = sum(s.product.price_inr for s in self.slots)


class AgentResponse(BaseModel):
    """Final orchestrator response."""
    message: str
    recommendations: list[Recommendation] = Field(default_factory=list)
    outfit_recommendations: list[OutfitRecommendation] = Field(default_factory=list)
    user_profile: UserProfile | None = None
    intent: str = ""


print("✓ Data models defined:",
      [m.__name__ for m in (
          UserProfile, Outfit, Product, ReviewSummary,
          Recommendation, OutfitRecommendation, AgentResponse,
      )])

## Section 4 — Vision Analysis Module

Analyze uploaded user photos with Qwen2.5-VL to extract skin tone, body type, and style.

In [ ]:
class VisionAnalysisResult(BaseModel):
    skin_tone: str
    body_type: str
    style: str
    confidence: float = Field(ge=0.0, le=1.0)
    color_recommendations: list[str] = Field(default_factory=list)
    styling_notes: str = ""


VISION_SYSTEM_PROMPT = """You are an expert fashion stylist and image analyst.
Analyze the person's photo and return ONLY valid JSON with these fields:
- skin_tone: warm | cool | neutral | deep | fair | medium
- body_type: athletic | slim | curvy | broad | petite | average
- style: brief description of their current fashion style
- confidence: float 0-1 for analysis confidence
- color_recommendations: list of 3-5 colors that suit them
- styling_notes: 1-2 sentences of personalized styling advice

Be respectful, professional, and focus on fashion styling only."""


def analyze_user_image(
    image: Image.Image | str | Path,
    gender: Gender = Gender.UNISEX,
) -> UserProfile:
    """
    Analyze an uploaded user image using Qwen2.5-VL.

    Args:
        image: PIL Image, file path, or URL
        gender: optional gender hint

    Returns:
        UserProfile populated from vision analysis
    """
    if isinstance(image, (str, Path)):
        image = Image.open(image)

    data_uri = image_to_base64(image)

    messages: list[dict[str, Any]] = [
        {"role": "system", "content": VISION_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": data_uri},
                },
                {
                    "type": "text",
                    "text": (
                        f"Analyze this person's appearance for fashion styling. "
                        f"Gender hint: {gender.value}. Return JSON only."
                    ),
                },
            ],
        },
    ]

    try:
        result = llm_json_call(
            client=vision_client,
            model=VISION_MODEL,
            messages=messages,
            schema_model=VisionAnalysisResult,
            temperature=0.2,
            max_tokens=1024,
        )
        logger.info(
            "Vision analysis: skin=%s body=%s style=%s conf=%.2f",
            result.skin_tone, result.body_type, result.style, result.confidence,
        )
        return UserProfile(
            skin_tone=result.skin_tone,
            body_type=result.body_type,
            style=result.style,
            confidence=result.confidence,
            gender=gender,
            color_preferences=result.color_recommendations,
            notes=result.styling_notes,
        )
    except Exception as exc:
        logger.error("Vision analysis failed: %s", exc)
        return UserProfile(
            skin_tone="neutral",
            body_type="average",
            style="casual contemporary",
            confidence=0.3,
            gender=gender,
            notes=f"Vision analysis unavailable: {exc}",
        )


class ProductStyleScore(BaseModel):
    """Vision-based suitability score for a product."""
    style_match: float = Field(ge=0.0, le=1.0, default=0.5)
    color_match: float = Field(ge=0.0, le=1.0, default=0.5)
    occasion_fit: float = Field(ge=0.0, le=1.0, default=0.5)
    reasoning: str = ""
    combined_score: float = Field(ge=0.0, le=1.0, default=0.5)


def fetch_product_image_url(product: Product) -> str:
    """Try to find a product image URL from page metadata."""
    if product.image_url:
        return product.image_url
    try:
        resp = requests.get(product.url, timeout=8, headers={
            "User-Agent": "Mozilla/5.0 (compatible; StylistBot/1.0)",
        })
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for sel in (
            ("meta", {"property": "og:image"}),
            ("meta", {"name": "og:image"}),
            ("img", {"id": "landingImage"}),
        ):
            if sel[0] == "meta":
                tag = soup.find("meta", sel[1])
                if tag and tag.get("content"):
                    return str(tag["content"])
            else:
                tag = soup.find("img", sel[1])
                if tag and tag.get("src"):
                    return str(tag["src"])
    except Exception as exc:
        logger.debug("Product image fetch failed: %s", exc)
    return ""


def score_product_with_vision(
    product: Product,
    user_profile: UserProfile | None = None,
    gender: Gender = Gender.UNISEX,
    occasion: str = "",
    user_photo: Image.Image | None = None,
) -> ProductStyleScore:
    """Use Qwen2.5-VL to score how well a product suits the user (requires product image)."""
    img_url = fetch_product_image_url(product)
    if not img_url:
        return ProductStyleScore(reasoning="No product image available for vision scoring")

    profile_txt = "No profile"
    if user_profile:
        profile_txt = (
            f"skin={user_profile.skin_tone}, body={user_profile.body_type}, "
            f"style={user_profile.style}, colors={user_profile.color_preferences}"
        )

    content: list[dict[str, Any]] = []
    if user_photo is not None:
        content.append({"type": "image_url", "image_url": {"url": image_to_base64(user_photo)}})
        content.append({"type": "text", "text": "Reference photo of the customer (for personalization)."})
    content.append({"type": "image_url", "image_url": {"url": img_url}})
    content.append({
        "type": "text",
        "text": (
            f"Score this product for the customer. Gender: {gender.value}. Occasion: {occasion}. "
            f"Profile: {profile_txt}. Product: {product.title}. "
            "Return JSON only: {style_match, color_match, occasion_fit, reasoning, combined_score} "
            "where combined_score is overall 0-1 suitability."
        ),
    })

    try:
        result = llm_json_call(
            client=vision_client,
            model=VISION_MODEL,
            messages=[
                {"role": "system", "content": "You are a fashion stylist. Score product suitability. Return JSON only."},
                {"role": "user", "content": content},
            ],
            schema_model=ProductStyleScore,
            temperature=0.2,
            max_tokens=512,
        )
        if result.combined_score == 0.5 and result.style_match != 0.5:
            result.combined_score = round(
                (result.style_match + result.color_match + result.occasion_fit) / 3, 2
            )
        return result
    except Exception as exc:
        logger.warning("Vision product scoring failed: %s", exc)
        return ProductStyleScore(reasoning=f"Vision scoring unavailable: {exc}")


print("✓ analyze_user_image() + score_product_with_vision() ready")

## Section 5 — Stylist Agent

Generate 3 complete outfit concepts for an occasion and budget using Qwen3-32B.

In [ ]:
class OutfitListResponse(BaseModel):
    outfits: list[Outfit]


FEMALE_ONLY_KEYWORDS = frozenset({
    "saree", "sari", "lehenga", "anarkali", "salwar", "kurti", "blouse", "dupatta", "lehenga-choli",
})
MALE_WEDDING_DEFAULTS: dict[str, str] = {
    "top": "men's maroon silk kurta sherwani for wedding",
    "bottom": "men's ivory churidar pajama for wedding",
    "footwear": "men's mojari jutti wedding ethnic shoes",
    "accessories": "men's wedding stole pocket square brooch set",
}
FEMALE_WEDDING_DEFAULTS: dict[str, str] = {
    "top": "women's designer ethnic kurta for wedding",
    "bottom": "women's palazzo or lehenga skirt for wedding",
    "footwear": "women's ethnic jutti heels for wedding",
    "accessories": "women's wedding jewelry clutch set",
}


def _gender_label(gender: Gender) -> str:
    return {"male": "MEN", "female": "WOMEN", "unisex": "UNISEX", "other": "UNISEX"}.get(gender.value, "UNISEX")


def _gender_rules(gender: Gender, occasion: str) -> str:
    if gender == Gender.MALE:
        return (
            "CRITICAL: Customer is MALE. Use ONLY men's clothing. "
            "For Indian wedding: sherwani, kurta-pajama, bandhgala, indo-western suit, waistcoat. "
            "NEVER use saree, lehenga, anarkali, salwar-kameez, kurti, or women's dupatta-set."
        )
    if gender == Gender.FEMALE:
        return (
            "CRITICAL: Customer is FEMALE. Use women's clothing only — "
            "saree, lehenga, anarkali, salwar suit, gown, etc. as appropriate."
        )
    return "Gender-neutral or unisex styling is acceptable."


def _sanitize_outfit_item(item: OutfitItem, gender: Gender, occasion: str) -> OutfitItem:
    """Replace female-only items when customer is male (and vice versa for obvious male-only)."""
    desc = item.description.lower()
    if gender == Gender.MALE and any(kw in desc for kw in FEMALE_ONLY_KEYWORDS):
        replacement = MALE_WEDDING_DEFAULTS.get(item.category, "men's wedding ethnic outfit piece")
        logger.warning("Replacing female item '%s' with '%s' for male customer", item.description[:50], replacement)
        return item.model_copy(update={"description": replacement})
    return item


def _sanitize_outfits(outfits: list[Outfit], gender: Gender, occasion: str) -> list[Outfit]:
    cleaned: list[Outfit] = []
    for outfit in outfits:
        items = [_sanitize_outfit_item(it, gender, occasion) for it in outfit.items]
        cleaned.append(outfit.model_copy(update={"items": items}))
    return cleaned


STYLIST_SYSTEM_PROMPT = """You are a professional fashion stylist based in India.
Generate exactly 3 distinct, complete outfit concepts as JSON.

Each outfit MUST include items for ALL categories:
- top
- bottom
- footwear
- accessories

Rules:
1. Stay within the total budget (INR) — distribute prices realistically across items
2. Match the occasion perfectly
3. STRICTLY match the customer's GENDER — this is mandatory
4. Personalize for the user's profile (skin tone, body type, style)
5. Use Indian market appropriate styles
6. Each item needs a detailed description suitable for product search (include "men's" or "women's" in every item)
7. Provide style_reasoning explaining WHY this outfit works

Return JSON: {"outfits": [{name, occasion, items: [{category, description, color, material, estimated_price_inr}], style_reasoning, total_estimated_budget_inr}]}
"""


def generate_outfits(
    occasion: str,
    budget: float,
    gender: Gender = Gender.UNISEX,
    city: str = "Mumbai",
    user_profile: UserProfile | None = None,
) -> list[Outfit]:
    """Generate 3 outfit concepts for the given occasion, budget, and gender."""
    profile_text = "No user profile available."
    if user_profile:
        profile_text = (
            f"Skin tone: {user_profile.skin_tone}, "
            f"Body type: {user_profile.body_type}, "
            f"Style: {user_profile.style}, "
            f"Colors: {user_profile.color_preferences}, "
            f"Notes: {user_profile.notes}"
        )

    user_prompt = f"""Create 3 complete outfits:
- Occasion: {occasion}
- Budget: ₹{budget:,.0f} INR (total per outfit)
- Gender: {_gender_label(gender)} ({gender.value}) — MUST follow gender rules below
- City: {city}
- User Profile: {profile_text}

{_gender_rules(gender, occasion)}

Every item description MUST start with "men's" or "women's" matching the gender above.
Return exactly 3 outfits as JSON."""

    messages = [
        {"role": "system", "content": STYLIST_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    try:
        response = llm_json_call(
            client=text_client,
            model=TEXT_MODEL,
            messages=messages,
            schema_model=OutfitListResponse,
            temperature=0.4,
            max_tokens=4096,
        )
        outfits = _sanitize_outfits(response.outfits[:3], gender, occasion)
        logger.info("Generated %d outfits for %s / %s (budget ₹%.0f)", len(outfits), occasion, gender.value, budget)
        return outfits
    except Exception as exc:
        logger.error("Outfit generation failed: %s", exc)
        defaults = MALE_WEDDING_DEFAULTS if gender == Gender.MALE else FEMALE_WEDDING_DEFAULTS
        return [
            Outfit(
                name=f"{occasion.title()} Look {i + 1}",
                occasion=occasion,
                items=[
                    OutfitItem(category="top", description=defaults["top"], estimated_price_inr=budget * 0.35),
                    OutfitItem(category="bottom", description=defaults["bottom"], estimated_price_inr=budget * 0.25),
                    OutfitItem(category="footwear", description=defaults["footwear"], estimated_price_inr=budget * 0.25),
                    OutfitItem(category="accessories", description=defaults["accessories"], estimated_price_inr=budget * 0.15),
                ],
                style_reasoning=f"Fallback {gender.value} outfit {i + 1}",
            )
            for i in range(3)
        ]


print("✓ generate_outfits() ready — gender-aware")


## Section 6 — Product Search Module

Discover products across Indian e-commerce sites using DuckDuckGo search.

In [ ]:
import time
import random
import logging
from urllib.parse import urlparse, quote_plus

import requests
from bs4 import BeautifulSoup

# Quiet noisy ddgs backend warnings (timeouts on brave/yandex/etc are normal)
logging.getLogger("ddgs").setLevel(logging.ERROR)
logging.getLogger("primp").setLevel(logging.ERROR)

try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS  # fallback

# Shared HTTP session — avoids unclosed socket warnings
_HTTP = requests.Session()
_HTTP.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-IN,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml",
})

ECOMMERCE_SITES: dict[str, str] = {
    "amazon": "amazon.in",
    "flipkart": "flipkart.com",
    "myntra": "myntra.com",
    "ajio": "ajio.com",
    "nykaa": "nykaa.com",
}
# Demo speed settings
FAST_OUTFIT_MODE = True          # skip per-product LLM review during outfit build (~10x faster)
USE_VISION_FOR_PRODUCTS = True  # score product images when user photo uploaded
OUTFIT_SEARCH_SITES = ["amazon", "myntra"]  # 2 sites instead of 5
SEARCH_TIMEOUT_SEC = 8



def _parse_price(text: str) -> float:
    if not text:
        return 0.0
    patterns = [
        r"[₹]\s*([\d,]+(?:\.\d{1,2})?)",
        r"(?:Rs\.?|INR)\s*([\d,]+(?:\.\d{1,2})?)",
        r"([\d,]+)\s*/-",
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            try:
                return float(match.group(1).replace(",", ""))
            except ValueError:
                continue
    return 0.0


def _parse_rating(text: str) -> tuple[float, int]:
    """Parse rating and review count safely — never crash on malformed snippets."""
    rating = 0.0
    count = 0
    if not text:
        return rating, count
    try:
        r_match = re.search(r"(\d(?:\.\d)?)\s*(?:out of 5|★|stars?|/5)", text, re.I)
        if r_match and r_match.group(1):
            rating = min(float(r_match.group(1)), 5.0)
    except (ValueError, TypeError):
        pass
    try:
        c_match = re.search(r"([\d,]+)\s*(?:ratings?|reviews?)", text, re.I)
        if c_match and c_match.group(1):
            raw = c_match.group(1).replace(",", "").strip()
            if raw.isdigit():
                count = int(raw)
    except (ValueError, TypeError):
        pass
    return rating, count


def _detect_source(url: str) -> str:
    domain = urlparse(url).netloc.lower()
    for source, site in ECOMMERCE_SITES.items():
        if site in domain:
            return source
    return "other"


def _ddg_search(query: str, max_results: int = 5) -> list[dict[str, str]]:
    """Fast DuckDuckGo search — single backend, short timeout."""
    try:
        with DDGS() as ddgs:
            kwargs: dict = {"max_results": max_results, "region": "in-en"}
            try:
                results = list(ddgs.text(query, backend="lite", timeout=SEARCH_TIMEOUT_SEC, **kwargs))
            except TypeError:
                results = list(ddgs.text(query, **kwargs))
        return results  # type: ignore[return-value]
    except Exception as exc:
        logger.debug("Search failed for '%s': %s", query[:50], exc)
        return []


def search_products(
    query: str,
    budget: float | None = None,
    sites: list[str] | None = None,
    max_per_site: int = 2,
    gender: Gender | None = None,
) -> list[Product]:
    """Search Indian e-commerce sites for products."""
    target_sites = sites or list(ECOMMERCE_SITES.keys())
    products: list[Product] = []
    seen_urls: set[str] = set()

    # Ensure gender appears in search query for better results
    q = query.strip()
    if gender == Gender.MALE and "men" not in q.lower():
        q = f"men's {q}"
    elif gender == Gender.FEMALE and not any(w in q.lower() for w in ("women", "women's", "ladies")):
        q = f"women's {q}"

    for site_key in target_sites:
        domain = ECOMMERCE_SITES.get(site_key)
        if not domain:
            continue

        search_query = f"{q} site:{domain}"
        if budget:
            search_query += f" under {int(budget)}"

        logger.info("Searching: %s", search_query[:100])
        try:
            results = _ddg_search(search_query, max_results=max_per_site + 1)
        except Exception as exc:
            logger.warning("Search failed for %s: %s", domain, exc)
            continue

        for result in results:
            url = result.get("href", result.get("link", ""))
            title = result.get("title", "")
            snippet = result.get("body", result.get("snippet", ""))
            if not url or url in seen_urls or domain not in url:
                continue
            seen_urls.add(url)
            price = _parse_price(f"{title} {snippet}")
            try:
                rating, review_count = _parse_rating(snippet)
            except Exception:
                rating, review_count = 0.0, 0
            if budget and price > 0 and price > budget * 1.15:
                continue
            products.append(Product(
                title=title[:200], price_inr=price, rating=rating,
                review_count=review_count, url=url, source=site_key, snippet=snippet[:500],
            ))
            if len([p for p in products if p.source == site_key]) >= max_per_site:
                break
        time.sleep(0.2)

    logger.info("Found %d products for '%s'", len(products), q[:60])
    return products


def fetch_product_reviews(url: str, max_reviews: int = 5) -> list[str]:
    reviews: list[str] = []
    try:
        resp = _HTTP.get(url, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for sel in ("[data-hook='review-body']", ".review-text", "[class*='review']"):
            for el in soup.select(sel)[:max_reviews * 2]:
                text = el.get_text(strip=True)
                if 20 < len(text) < 2000:
                    reviews.append(text)
                if len(reviews) >= max_reviews:
                    break
            if reviews:
                break
    except Exception as exc:
        logger.debug("Review fetch failed for %s: %s", url[:60], exc)
    return reviews[:max_reviews]


print("✓ search_products() ready — robust DDG + gender-aware queries")


## Section 7 — Review Analysis Module

Summarize product reviews using Qwen3-32B for pros, cons, and sentiment.

In [ ]:
REVIEW_SYSTEM_PROMPT = """You are a product review analyst for Indian e-commerce.
Analyze the provided reviews and return ONLY valid JSON:
{
  "pros": ["list of positive points"],
  "cons": ["list of negative points"],
  "overall_sentiment": "positive" | "neutral" | "negative" | "mixed",
  "summary": "2-3 sentence summary",
  "review_quality_score": 0.0-1.0
}

review_quality_score reflects how detailed, consistent, and trustworthy the reviews are.
Be objective and concise."""


def analyze_reviews(
    raw_reviews: list[str],
    product_title: str = "",
    product_snippet: str = "",
) -> ReviewSummary:
    """
    Summarize raw review text into structured pros/cons/sentiment.

    Args:
        raw_reviews: List of review text strings
        product_title: Product name for context
        product_snippet: Search snippet as fallback context

    Returns:
        ReviewSummary with pros, cons, sentiment, quality score
    """
    if not raw_reviews and not product_snippet:
        return ReviewSummary(
            pros=["No review data available"],
            cons=[],
            overall_sentiment="neutral",
            summary="Insufficient review data to analyze.",
            review_quality_score=0.2,
        )

    review_text = "\n---\n".join(raw_reviews[:8]) if raw_reviews else product_snippet

    messages = [
        {"role": "system", "content": REVIEW_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Product: {product_title}\n\n"
                f"Reviews:\n{review_text[:3000]}"
            ),
        },
    ]

    try:
        return llm_json_call(
            client=text_client,
            model=TEXT_MODEL,
            messages=messages,
            schema_model=ReviewSummary,
            temperature=0.2,
            max_tokens=1024,
        )
    except Exception as exc:
        logger.warning("Review analysis failed for '%s': %s", product_title, exc)
        return ReviewSummary(
            pros=["Product available on trusted platform"] if product_title else [],
            cons=["Limited review data"],
            overall_sentiment="neutral",
            summary=product_snippet[:200] or "Review analysis unavailable.",
            review_quality_score=0.3,
        )


def enrich_products_with_reviews(products: list[Product]) -> list[Product]:
    """Fetch and attach raw reviews to product objects."""
    enriched: list[Product] = []
    for product in products:
        reviews = fetch_product_reviews(product.url)
        if not reviews and product.snippet:
            reviews = [product.snippet]
        enriched.append(product.model_copy(update={"raw_reviews": reviews}))
    return enriched


print("✓ analyze_reviews() ready")

## Section 8 — Product Ranking Module

Score and rank products using weighted factors:
- Rating (40%) · Review Quality (30%) · Budget Fit (20%) · Style Relevance (10%)

In [ ]:
# Ranking weights (must sum to 1.0)
RANKING_WEIGHTS = {
    "rating": 0.40,
    "review_quality": 0.30,
    "budget_fit": 0.20,
    "style_relevance": 0.10,
}


def _score_rating(product: Product) -> float:
    """Normalize rating to 0-100. Bonus for high review count."""
    base = (product.rating / 5.0) * 100 if product.rating > 0 else 40.0
    if product.review_count > 1000:
        base = min(base + 10, 100)
    elif product.review_count > 100:
        base = min(base + 5, 100)
    return base


def _score_budget_fit(price: float, budget: float) -> float:
    """Score how well price fits budget (100 = perfect, 0 = way over)."""
    if budget <= 0:
        return 50.0
    if price <= 0:
        return 60.0  # unknown price — neutral-positive
    ratio = price / budget
    if ratio <= 0.5:
        return 90.0  # great value
    if ratio <= 0.85:
        return 100.0  # ideal range
    if ratio <= 1.0:
        return 85.0
    if ratio <= 1.15:
        return 50.0
    return max(0.0, 30.0 - (ratio - 1.15) * 100)


def _score_style_relevance(
    product: Product,
    query: str,
    user_profile: UserProfile | None = None,
) -> float:
    """Heuristic style relevance based on query keyword overlap."""
    query_tokens = set(query.lower().split())
    title_tokens = set(product.title.lower().split())
    overlap = len(query_tokens & title_tokens)
    base = min(overlap * 20, 80)

    if user_profile and user_profile.color_preferences:
        for color in user_profile.color_preferences:
            if color.lower() in product.title.lower():
                base = min(base + 10, 100)
    return max(base, 30.0)


def _score_review_quality(summary: ReviewSummary | None) -> float:
    """Convert review quality score to 0-100."""
    if not summary:
        return 40.0
    sentiment_bonus = {
        "positive": 15, "mixed": 5, "neutral": 0, "negative": -15,
    }
    base = summary.review_quality_score * 100
    base += sentiment_bonus.get(summary.overall_sentiment, 0)
    return max(0.0, min(base, 100.0))


def rank_products(
    products: list[Product],
    budget: float,
    query: str = "",
    user_profile: UserProfile | None = None,
    review_summaries: dict[str, ReviewSummary] | None = None,
    top_k: int = 10,
    vision_scores: dict[str, float] | None = None,
) -> list[RankedProduct]:
    """
    Rank products by composite score.

    Args:
        products: Candidate products
        budget: Budget in INR
        query: Original search query for style relevance
        user_profile: Optional user profile
        review_summaries: Map of product URL → ReviewSummary
        top_k: Number of top products to return

    Returns:
        Sorted list of RankedProduct (highest score first)
    """
    review_summaries = review_summaries or {}
    ranked: list[RankedProduct] = []

    for product in products:
        summary = review_summaries.get(product.url)
        vis_bonus = (vision_scores or {}).get(product.url, 0.0) * 15.0  # up to +15 pts
        breakdown = {
            "rating": _score_rating(product) * RANKING_WEIGHTS["rating"],
            "review_quality": _score_review_quality(summary) * RANKING_WEIGHTS["review_quality"],
            "budget_fit": _score_budget_fit(product.price_inr, budget) * RANKING_WEIGHTS["budget_fit"],
            "style_relevance": _score_style_relevance(product, query, user_profile) * RANKING_WEIGHTS["style_relevance"],
        }
        total = sum(breakdown.values()) + vis_bonus

        ranked.append(RankedProduct(
            product=product,
            review_summary=summary,
            ranking_score=round(total, 2),
            score_breakdown={k: round(v, 2) for k, v in breakdown.items()},
        ))

    ranked.sort(key=lambda r: r.ranking_score, reverse=True)
    logger.info("Ranked %d products, top score: %.1f", len(ranked), ranked[0].ranking_score if ranked else 0)
    return ranked[:top_k]


print("✓ rank_products() ready | weights:", RANKING_WEIGHTS)

## Section 9 — Shopping Agent

End-to-end product recommendation: search → review → rank → explain.

In [ ]:
def _generate_recommendation_explanation(
    ranked: RankedProduct,
    query: str,
    budget: float,
    rank: int,
) -> str:
    """Use LLM to explain why a product was recommended."""
    summary = ranked.review_summary
    prompt = f"""As a professional fashion stylist, explain in 2-3 sentences why this is rank #{rank} recommendation.

Query: {query}
Budget: ₹{budget:,.0f}
Product: {ranked.product.title}
Price: ₹{ranked.product.price_inr:,.0f}
Rating: {ranked.product.rating}/5 ({ranked.product.review_count} reviews)
Source: {ranked.product.source}
Score: {ranked.ranking_score}/100
Pros: {summary.pros if summary else 'N/A'}
Cons: {summary.cons if summary else 'N/A'}

Be specific about value, quality, and style fit. Return plain text only."""

    try:
        response = text_client.chat.completions.create(
            model=TEXT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.4,
            max_tokens=200,
            **TEXT_CHAT_KWARGS,
        )
        return (response.choices[0].message.content or "").strip()
    except Exception as exc:
        logger.warning("Explanation generation failed: %s", exc)
        return (
            f"Ranked #{rank} with score {ranked.ranking_score:.0f}/100. "
            f"Rated {ranked.product.rating}/5 on {ranked.product.source}."
        )


def recommend_products(
    query: str,
    budget: float,
    user_profile: UserProfile | None = None,
    top_k: int = 3,
) -> list[Recommendation]:
    """
    Full product recommendation pipeline.

    Process: search → fetch reviews → analyze → rank → explain

    Args:
        query: Product search query (e.g. "running shoes")
        budget: Max budget in INR
        user_profile: Optional personalization context
        top_k: Number of recommendations (default 3)

    Returns:
        Top recommendations with explanations
    """
    logger.info("Shopping agent: query='%s' budget=₹%.0f", query, budget)

    # 1. Search products
    gender = user_profile.gender if user_profile else None
    products = search_products(query, budget=budget, gender=gender)
    if not products:
        logger.warning("No products found for '%s'", query)
        return []

    # 2. Enrich with reviews
    products = enrich_products_with_reviews(products[:12])

    # 3. Analyze reviews
    review_map: dict[str, ReviewSummary] = {}
    for product in products:
        summary = analyze_reviews(
            product.raw_reviews,
            product_title=product.title,
            product_snippet=product.snippet,
        )
        review_map[product.url] = summary

    # 4. Rank
    ranked = rank_products(
        products, budget=budget, query=query,
        user_profile=user_profile, review_summaries=review_map, top_k=top_k,
    )

    # 5. Generate explanations
    recommendations: list[Recommendation] = []
    for i, rp in enumerate(ranked, start=1):
        explanation = _generate_recommendation_explanation(rp, query, budget, i)
        recommendations.append(Recommendation(
            ranked_product=rp,
            rank=i,
            explanation=explanation,
        ))

    logger.info("Returning %d recommendations", len(recommendations))
    return recommendations


print("✓ recommend_products() ready")

## Section 10 — Outfit Shopping Agent

Build 3 complete purchasable outfit boards: generate → search → rank → assemble.

In [ ]:
def _heuristic_review_summary(product: Product) -> ReviewSummary:
    """Fast review summary without LLM — used in outfit build mode."""
    sentiment = "positive" if product.rating >= 4.0 else "neutral"
    return ReviewSummary(
        pros=[product.snippet[:120] or f"Listed on {product.source}"],
        cons=["Detailed reviews skipped in fast mode"] if not product.snippet else [],
        overall_sentiment=sentiment,
        summary=product.snippet[:200] or product.title,
        review_quality_score=min(0.3 + product.rating / 10, 0.8),
    )


def _search_outfit_item(
    item: OutfitItem,
    slot_budget: float,
    gender: Gender = Gender.UNISEX,
    user_profile: UserProfile | None = None,
    occasion: str = "wedding",
    user_photo: Image.Image | None = None,
) -> OutfitProductSlot | None:
    """Search and rank products for one outfit slot (optimized for speed)."""
    item = _sanitize_outfit_item(item, gender, occasion)
    query = f"{item.color} {item.description}".strip() if item.color else item.description

    products = search_products(
        query, budget=slot_budget, sites=OUTFIT_SEARCH_SITES,
        max_per_site=1, gender=gender,
    )
    if not products:
        short = " ".join(item.description.split()[:4])
        products = search_products(short, budget=slot_budget, sites=OUTFIT_SEARCH_SITES[:1], max_per_site=1, gender=gender)
    if not products:
        logger.warning("No products for slot: %s", item.description[:60])
        return None

    review_map = {p.url: _heuristic_review_summary(p) for p in products}

    vision_scores: dict[str, float] = {}
    if USE_VISION_FOR_PRODUCTS and user_photo is not None and products:
        top = products[0]
        vs = score_product_with_vision(top, user_profile, gender, occasion, user_photo)
        vision_scores[top.url] = vs.combined_score
        logger.info("Vision score %.2f for %s", vs.combined_score, top.title[:40])

    ranked = rank_products(
        products, budget=slot_budget, query=item.description,
        user_profile=user_profile, review_summaries=review_map,
        vision_scores=vision_scores or None, top_k=1,
    )
    if not ranked:
        return None
    best = ranked[0]
    return OutfitProductSlot(category=item.category, product=best.product, review_summary=best.review_summary, ranking_score=best.ranking_score)


def build_complete_outfits(
    occasion: str,
    budget: float,
    gender: Gender = Gender.UNISEX,
    city: str = "Mumbai",
    user_profile: UserProfile | None = None,
    user_photo: Image.Image | None = None,
) -> list[OutfitRecommendation]:
    """Build purchasable outfit boards (fast mode: ~2-4 min vs 10+ min)."""
    logger.info("Outfit agent: occasion=%s budget=₹%.0f gender=%s fast=%s", occasion, budget, gender.value, FAST_OUTFIT_MODE)

    outfits = generate_outfits(occasion=occasion, budget=budget, gender=gender, city=city, user_profile=user_profile)
    outfit_recommendations: list[OutfitRecommendation] = []

    for outfit in outfits:
        slots: list[OutfitProductSlot] = []
        n_items = max(len(outfit.items), 1)
        for item in outfit.items:
            slot_budget = item.estimated_price_inr or (budget / n_items)
            try:
                slot = _search_outfit_item(
                    item, slot_budget, gender=gender, user_profile=user_profile,
                    occasion=occasion, user_photo=user_photo,
                )
                if slot:
                    slots.append(slot)
            except Exception as exc:
                logger.warning("Slot failed: %s", exc)
        outfit_recommendations.append(OutfitRecommendation(
            outfit=outfit,
            slots=slots,
            total_price_inr=sum(s.product.price_inr for s in slots),
            stylist_notes=(
                f"{outfit.style_reasoning}\n\nGender: {gender.value} | "
                f"Filled {len(slots)}/{len(outfit.items)} slots. "
                f"{'Vision personalization ON' if user_photo else 'Upload photo for vision scoring'}."
            ),
        ))

    logger.info("Built %d outfit boards", len(outfit_recommendations))
    return outfit_recommendations


print("✓ build_complete_outfits() ready — fast mode + optional vision scoring")


## Section 11 — Agent Orchestrator

Lightweight manual orchestration: intent classification → tool routing → response assembly.

```
User → Orchestrator → [Vision Tool] → Stylist Tool → Product Search → Review Analysis → Ranking → Response
```

In [ ]:
def _parse_gender(value: str, default: Gender) -> Gender:
    try:
        return Gender(value.lower())
    except ValueError:
        return default


class IntentClassification(BaseModel):
    intent: Literal["product_search", "outfit_build", "general_chat"]
    query: str = ""
    occasion: str = ""
    budget: float = 0.0
    gender: str = "unisex"
    reasoning: str = ""

    @classmethod
    def model_validate(cls, obj):  # type: ignore[override]
        if isinstance(obj, dict):
            obj = {
                **obj,
                "query": obj.get("query") or "",
                "occasion": obj.get("occasion") or "",
                "reasoning": obj.get("reasoning") or "",
                "gender": obj.get("gender") or "unisex",
                "budget": float(obj.get("budget") or 0.0),
            }
        return super().model_validate(obj)


INTENT_SYSTEM_PROMPT = """You are an intent classifier for a fashion shopping assistant.
Classify the user message into one of:
- product_search: user wants specific products (shoes, shirt, bag, etc.)
- outfit_build: user wants complete outfits for an occasion/event
- general_chat: general fashion advice or greeting

Extract:
- query: product search terms (for product_search)
- occasion: event type (for outfit_build) — wedding, party, office, casual, sports, etc.
- budget: numeric budget in INR (extract from text like "3000", "15k", "15000")
- gender: male/female/unisex if mentioned

Return ONLY JSON: {"intent", "query", "occasion", "budget", "gender", "reasoning"}"""


class StylistOrchestrator:
    """
    Main agent orchestrator — routes user input through the tool pipeline.

    Pipeline:
        User → classify intent → [vision] → stylist/search → review → rank → respond
    """

    def __init__(self) -> None:
        self.user_profile: UserProfile | None = None
        self.conversation_history: list[dict[str, str]] = []

    def _classify_intent(
        self,
        message: str,
        budget_override: float | None = None,
        occasion_override: str | None = None,
        gender: Gender = Gender.UNISEX,
    ) -> IntentClassification:
        """Classify user intent using Qwen3-32B."""
        messages = [
            {"role": "system", "content": INTENT_SYSTEM_PROMPT},
            {"role": "user", "content": f"UI gender selected: {gender.value}\n\nUser message: {message}"},
        ]
        try:
            intent = llm_json_call(
                client=text_client,
                model=TEXT_MODEL,
                messages=messages,
                schema_model=IntentClassification,
                temperature=0.1,
                max_tokens=512,
            )
            if budget_override and budget_override > 0:
                intent.budget = budget_override
            if occasion_override:
                intent.occasion = occasion_override
            return intent
        except Exception as exc:
            logger.warning("Intent classification failed: %s — using heuristics", exc)
            return self._heuristic_intent(message, budget_override, occasion_override, gender)

    def _heuristic_intent(
        self,
        message: str,
        budget: float | None,
        occasion: str | None,
        gender: Gender = Gender.UNISEX,
    ) -> IntentClassification:
        """Fallback keyword-based intent detection."""
        msg = message.lower()
        outfit_keywords = {"outfit", "wedding", "party", "wear", "dress", "look", "attending"}
        intent_type: Literal["product_search", "outfit_build", "general_chat"] = "product_search"

        if any(kw in msg for kw in outfit_keywords):
            intent_type = "outfit_build"

        # Extract budget from text
        extracted_budget = budget or 0.0
        if not extracted_budget:
            for pat in [r"(\d+)\s*k", r"under\s*(\d+)", r"budget\s*(\d+)", r"₹\s*([\d,]+)"]:
                m = re.search(pat, msg)
                if m:
                    val = m.group(1).replace(",", "")
                    extracted_budget = float(val)
                    if "k" in pat:
                        extracted_budget *= 1000
                    break

        return IntentClassification(
            intent=intent_type,
            query=message,
            occasion=occasion or ("wedding" if "wedding" in msg else "casual"),
            budget=extracted_budget or 5000.0,
            gender=gender.value,
            reasoning="Heuristic fallback",
        )

    def process(
        self,
        message: str,
        image: Image.Image | None = None,
        budget: float | None = None,
        occasion: str | None = None,
        gender: Gender = Gender.UNISEX,
        city: str = "Mumbai",
    ) -> AgentResponse:
        """
        Main entry point — process user message through the agent pipeline.

        Args:
            message: User chat message
            image: Optional uploaded photo
            budget: Optional budget override (INR)
            occasion: Optional occasion override
            gender: User gender
            city: User city

        Returns:
            AgentResponse with recommendations and/or outfit boards
        """
        logger.info("Orchestrator processing: '%s' | gender=%s", message[:100], gender.value)
        self.conversation_history.append({"role": "user", "content": message})

        # UI gender always wins — store on profile for downstream agents
        if self.user_profile is None:
            self.user_profile = UserProfile(
                skin_tone="neutral", body_type="average", style="classic",
                gender=gender, confidence=0.5,
            )
        else:
            self.user_profile = self.user_profile.model_copy(update={"gender": gender})

        # Step 1: Vision analysis (if image provided)
        if image is not None:
            logger.info("Running vision analysis on uploaded image")
            self.user_profile = analyze_user_image(image, gender=gender)
            if city:
                self.user_profile.city = city

        # Step 2: Classify intent
        intent = self._classify_intent(message, budget, occasion, gender=gender)
        logger.info("Intent: %s | budget=₹%.0f | occasion=%s", intent.intent, intent.budget, intent.occasion)

        # Step 3: Route to appropriate agent
        if intent.intent == "product_search":
            recommendations = recommend_products(
                query=intent.query or message,
                budget=intent.budget,
                user_profile=self.user_profile,
                top_k=3,
            )
            msg = self._format_product_response(recommendations, intent)
            response = AgentResponse(
                message=msg,
                recommendations=recommendations,
                user_profile=self.user_profile,
                intent=intent.intent,
            )

        elif intent.intent == "outfit_build":
            occ = intent.occasion or occasion or "casual"
            outfit_recs = build_complete_outfits(
                occasion=occ,
                budget=intent.budget,
                gender=gender,
                city=city,
                user_profile=self.user_profile,
                user_photo=image,
            )
            msg = self._format_outfit_response(outfit_recs, intent)
            response = AgentResponse(
                message=msg,
                outfit_recommendations=outfit_recs,
                user_profile=self.user_profile,
                intent=intent.intent,
            )

        else:
            msg = self._general_chat(message)
            response = AgentResponse(
                message=msg,
                user_profile=self.user_profile,
                intent="general_chat",
            )

        self.conversation_history.append({"role": "assistant", "content": response.message})
        return response

    def _format_product_response(
        self,
        recommendations: list[Recommendation],
        intent: IntentClassification,
    ) -> str:
        if not recommendations:
            return (
                f"I couldn't find products matching '{intent.query}' "
                f"under ₹{intent.budget:,.0f}. Try adjusting your budget or search terms."
            )
        lines = [f"Here are my top {len(recommendations)} recommendations for **{intent.query}** (budget: ₹{intent.budget:,.0f}):\n"]
        for rec in recommendations:
            p = rec.ranked_product.product
            price_str = f"₹{p.price_inr:,.0f}" if p.price_inr > 0 else "Price on site"
            lines.append(
                f"**#{rec.rank}. {p.title}**\n"
                f"   {price_str} | ⭐ {p.rating}/5 ({p.review_count} reviews) | {p.source}\n"
                f"   {rec.explanation}\n"
                f"   🔗 {p.url}\n"
            )
        return "\n".join(lines)

    def _format_outfit_response(
        self,
        outfits: list[OutfitRecommendation],
        intent: IntentClassification,
    ) -> str:
        if not outfits:
            return f"I couldn't build outfits for {intent.occasion}. Please try again."
        lines = [f"I've curated **{len(outfits)} complete {intent.occasion} outfits** within ₹{intent.budget:,.0f}:\n"]
        for i, outfit_rec in enumerate(outfits, 1):
            lines.append(f"### Outfit {i}: {outfit_rec.outfit.name}")
            lines.append(f"_{outfit_rec.outfit.style_reasoning}_\n")
            for slot in outfit_rec.slots:
                p = slot.product
                price_str = f"₹{p.price_inr:,.0f}" if p.price_inr > 0 else "Price on site"
                lines.append(f"- **{slot.category.title()}**: {p.title} — {price_str}")
            lines.append(f"**Total: ₹{outfit_rec.total_price_inr:,.0f}**\n")
        return "\n".join(lines)

    def _general_chat(self, message: str) -> str:
        try:
            response = text_client.chat.completions.create(
                model=TEXT_MODEL,
                messages=[
                    {"role": "system", "content": (
                        "You are a professional fashion stylist and shopping assistant for the Indian market. "
                        "Be helpful, personalized, and mention you can search products or build outfits."
                    )},
                    *self.conversation_history[-6:],
                ],
                temperature=0.6,
                max_tokens=512,
                **TEXT_CHAT_KWARGS,
            )
            return (response.choices[0].message.content or "").strip()
        except Exception as exc:
            return f"I'm your AI stylist! Ask me to find products or build outfits. (Error: {exc})"


orchestrator = StylistOrchestrator()
print("✓ StylistOrchestrator ready")

## Section 12 — Gradio Interface

Modern chat UI with image upload, budget/occasion inputs, product cards, and outfit boards.

> **Gradio budget fix:** Use `gr.Textbox` for budget — NOT `gr.Number(minimum=500)`.
> The Number field crashes with `Value 5 is less than minimum value 500` when the field is empty or partially edited.

In [ ]:
import gradio as gr

# ── HTML card renderers ──────────────────────────────────────────────

def _product_card_html(rec: Recommendation) -> str:
    p = rec.ranked_product.product
    rs = rec.ranked_product.review_summary
    price = f"₹{p.price_inr:,.0f}" if p.price_inr > 0 else "Check site"
    pros = ", ".join(rs.pros[:3]) if rs else ""
    cons = ", ".join(rs.cons[:2]) if rs else ""
    stars = "⭐" * int(p.rating) if p.rating > 0 else "No rating"

    return f"""
    <div style="border:1px solid #e0e0e0;border-radius:12px;padding:16px;margin:10px 0;
                background:linear-gradient(135deg,#fafafa,#fff);box-shadow:0 2px 8px rgba(0,0,0,.06);">
      <div style="display:flex;justify-content:space-between;align-items:center;">
        <span style="background:#6366f1;color:#fff;padding:2px 10px;border-radius:20px;font-size:12px;">
          #{rec.rank} Pick
        </span>
        <span style="color:#888;font-size:12px;text-transform:uppercase;">{p.source}</span>
      </div>
      <h3 style="margin:10px 0 6px;font-size:15px;color:#1a1a2e;">{p.title[:120]}</h3>
      <div style="display:flex;gap:16px;margin:8px 0;">
        <span style="font-size:20px;font-weight:700;color:#6366f1;">{price}</span>
        <span>{stars} {p.rating}/5 ({p.review_count})</span>
        <span style="color:#888;">Score: {rec.ranked_product.ranking_score:.0f}/100</span>
      </div>
      <p style="color:#444;font-size:13px;margin:8px 0;">{rec.explanation}</p>
      {"<p style='color:#16a34a;font-size:12px;'>✓ " + pros + "</p>" if pros else ""}
      {"<p style='color:#dc2626;font-size:12px;'>✗ " + cons + "</p>" if cons else ""}
      <a href="{p.url}" target="_blank"
         style="display:inline-block;margin-top:8px;padding:8px 20px;background:#6366f1;
                color:#fff;border-radius:8px;text-decoration:none;font-size:13px;">
        Buy on {p.source.title()} →
      </a>
    </div>"""


def _outfit_board_html(outfit_rec: OutfitRecommendation, index: int) -> str:
    slots_html = ""
    for slot in outfit_rec.slots:
        p = slot.product
        price = f"₹{p.price_inr:,.0f}" if p.price_inr > 0 else "Check site"
        img = p.image_url or "https://via.placeholder.com/80x80?text=Item"
        slots_html += f"""
        <div style="display:flex;gap:12px;align-items:center;padding:8px 0;
                    border-bottom:1px solid #f0f0f0;">
          <img src="{img}" width="60" height="60"
               style="border-radius:8px;object-fit:cover;background:#f5f5f5;"
               onerror="this.src='https://via.placeholder.com/60x60?text={slot.category}'"/>
          <div style="flex:1;">
            <div style="font-size:11px;color:#888;text-transform:uppercase;">{slot.category}</div>
            <div style="font-size:13px;font-weight:600;">{p.title[:80]}</div>
            <div style="font-size:13px;color:#6366f1;">{price}</div>
          </div>
          <a href="{p.url}" target="_blank"
             style="padding:4px 12px;border:1px solid #6366f1;color:#6366f1;
                    border-radius:6px;text-decoration:none;font-size:11px;">Buy</a>
        </div>"""

    return f"""
    <div style="border:2px solid #6366f1;border-radius:16px;padding:20px;margin:12px 0;
                background:#fff;box-shadow:0 4px 16px rgba(99,102,241,.12);">
      <div style="display:flex;justify-content:space-between;align-items:center;">
        <h2 style="margin:0;color:#1a1a2e;font-size:18px;">
          Outfit {index}: {outfit_rec.outfit.name}
        </h2>
        <span style="background:#6366f1;color:#fff;padding:4px 14px;border-radius:20px;font-size:13px;">
          ₹{outfit_rec.total_price_inr:,.0f}
        </span>
      </div>
      <p style="color:#666;font-size:13px;font-style:italic;margin:8px 0 12px;">
        {outfit_rec.outfit.style_reasoning}
      </p>
      {slots_html}
    </div>"""


def _render_results_html(response: AgentResponse) -> str:
    html = ""
    if response.user_profile and response.user_profile.confidence > 0.3:
        up = response.user_profile
        html += f"""
        <div style="background:#f0fdf4;border:1px solid #bbf7d0;border-radius:10px;
                    padding:12px;margin-bottom:12px;">
          <strong>👤 Your Style Profile:</strong>
          Skin: {up.skin_tone} | Body: {up.body_type} | Style: {up.style}
          (confidence: {up.confidence:.0%})
        </div>"""

    for rec in response.recommendations:
        html += _product_card_html(rec)

    for i, outfit_rec in enumerate(response.outfit_recommendations, 1):
        html += _outfit_board_html(outfit_rec, i)

    return html or "<p>No results to display.</p>"


# ── Gradio handler ───────────────────────────────────────────────────

def _resolve_budget(budget: float | int | str | None, message: str) -> float | None:
    """Use UI budget if valid; otherwise extract from chat message."""
    try:
        if budget is not None and str(budget).strip() != "":
            b = float(budget)
            if b >= 100:
                return b
    except (TypeError, ValueError):
        pass
    msg = message.lower()
    for pat, mult in [
        (r"(\d+)\s*k\b", 1000), (r"budget\s*(?:is\s*)?(?:₹?\s*)?([\d,]+)", 1),
        (r"under\s*(?:₹?\s*)?([\d,]+)", 1), (r"₹\s*([\d,]+)", 1), (r"\b([\d]{4,})\b", 1),
    ]:
        m = re.search(pat, msg)
        if m:
            val = float(m.group(1).replace(",", ""))
            return val * mult if mult > 1 else val
    return None


# ── Gradio chat format (messages API — required by Gradio 4.40+ / 5.x) ──
CHAT_USES_MESSAGES = True
try:
    gr.Chatbot(type="messages")
except TypeError:
    CHAT_USES_MESSAGES = False
print(f"Gradio {gr.__version__} — chat history format: {'messages' if CHAT_USES_MESSAGES else 'tuples'}")


def _history_to_messages(history: list) -> list[dict[str, str]]:
    if not history:
        return []
    if isinstance(history[0], dict):
        return [
            {"role": m.get("role", "user"), "content": str(m.get("content", ""))}
            for m in history
            if isinstance(m, dict)
        ]
    msgs: list[dict[str, str]] = []
    for turn in history:
        if isinstance(turn, (list, tuple)) and len(turn) >= 2:
            msgs.append({"role": "user", "content": str(turn[0] or "")})
            msgs.append({"role": "assistant", "content": str(turn[1] or "")})
    return msgs


def _append_history(history: list, user_msg: str, bot_msg: str) -> list:
    user_text = str(user_msg or "")
    bot_text = str(bot_msg or "")
    if CHAT_USES_MESSAGES:
        h = _history_to_messages(history or [])
        return h + [
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": bot_text},
        ]
    pairs: list = []
    if history:
        if isinstance(history[0], dict):
            msgs = _history_to_messages(history)
            for i in range(0, len(msgs) - 1, 2):
                pairs.append([msgs[i]["content"], msgs[i + 1]["content"]])
        else:
            pairs = [[str(a or ""), str(b or "")] for a, b in history]
    return pairs + [[user_text, bot_text]]


def chat_handler(
    message: str,
    history: list,
    image: Image.Image | None,
    budget: float | int | str | None,
    occasion: str,
    gender: str,
    city: str,
) -> tuple[list, str]:
    """Process user input through the orchestrator and update UI."""
    if not message.strip():
        return _history_to_messages(history or []), "<p>Please enter a message.</p>"

    try:
        gender_enum = Gender(gender)
        resolved_budget = _resolve_budget(budget, message)
        response = orchestrator.process(
            message=message,
            image=image,
            budget=resolved_budget,
            occasion=occasion if occasion else None,
            gender=gender_enum,
            city=city,
        )
        chat_reply = str(response.message or "Done — see outfit boards below.")
        if len(chat_reply) > 800:
            chat_reply = chat_reply[:800] + "\n\n…(full details in the results panel below)"
        history = _append_history(history, message, chat_reply)
        results_html = _render_results_html(response)
        return history, results_html
    except Exception as exc:
        logger.exception("Chat handler error")
        error_msg = f"Sorry, something went wrong: {exc}"
        history = _append_history(history, message, error_msg)
        return history, f"<p style='color:red;'>{error_msg}</p>"


# ── Build Gradio App ─────────────────────────────────────────────────

CUSTOM_CSS = """
.gradio-container { max-width: 1200px !important; margin: auto; }
#results-pane { min-height: 400px; }
footer { display: none !important; }
"""

with gr.Blocks(title="AI Personal Stylist") as demo:
    gr.Markdown("""
    # 👗 AI Personal Stylist & Shopping Agent
    *Powered by Qwen3-32B + Qwen2.5-VL on AMD MI300X via vLLM*

    Ask for product recommendations, complete outfits, or upload your photo for personalized styling.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(label="Upload Your Photo (optional)", type="pil", height=250)
            budget_input = gr.Textbox(
                label="Budget (₹ INR) — optional if mentioned in chat",
                value="15000",
                placeholder="e.g. 15000",
            )
            occasion_input = gr.Dropdown(
                label="Occasion",
                choices=["", "casual", "wedding", "party", "office", "sports", "formal", "date", "festival"],
                value="",
            )
            gender_input = gr.Dropdown(label="Gender", choices=["unisex", "male", "female"], value="unisex")
            city_input = gr.Textbox(label="City", value="Mumbai")

        with gr.Column(scale=2):
            if CHAT_USES_MESSAGES:
                chatbot = gr.Chatbot(label="Stylist Chat", height=400, type="messages")
            else:
                chatbot = gr.Chatbot(label="Stylist Chat", height=400)
            msg_input = gr.Textbox(
                label="Your Message",
                placeholder="e.g. recommend running shoes under 3000 | I am attending my friend's wedding budget 15000",
                lines=2,
            )
            with gr.Row():
                send_btn = gr.Button("✨ Get Recommendations", variant="primary", scale=3)
                clear_btn = gr.Button("Clear", scale=1)

    results_html = gr.HTML(label="Results", elem_id="results-pane")

    gr.Examples(
        examples=[
            ["recommend running shoes under 3000", "3000", "", "unisex"],
            ["I am attending my friend's wedding, budget 15000", "15000", "wedding", "female"],
            ["find me a casual linen shirt under 2000", "2000", "casual", "male"],
            ["suggest formal office wear under 8000", "8000", "office", "female"],
        ],
        inputs=[msg_input, budget_input, occasion_input, gender_input],
        label="Try these examples",
    )

    # Wire events
    inputs = [msg_input, chatbot, image_input, budget_input, occasion_input, gender_input, city_input]
    outputs = [chatbot, results_html]

    send_btn.click(chat_handler, inputs=inputs, outputs=outputs).then(
        lambda: "", outputs=msg_input
    )
    msg_input.submit(chat_handler, inputs=inputs, outputs=outputs).then(
        lambda: "", outputs=msg_input
    )
    clear_btn.click(lambda: ([], "<p></p>", None), outputs=[chatbot, results_html, image_input])

print("✓ Gradio app built — run demo.launch() in the next cell")

### Gradio dev workflow (no full kernel restart needed)

| What you changed | Re-run these cells only |
|------------------|-------------------------|
| Agent / search / ranking code | Changed cells + **Stop & Launch** below |
| Gradio UI layout | Gradio UI cell + **Stop & Launch** below |
| Port stuck / weird state | Run `stop_gradio()` or **Kernel → Interrupt**, then relaunch |

You do **not** need to re-run Sections 1–11 every time unless you restarted the kernel.

In [ ]:
import socket
import subprocess

GRADIO_PORT: int | None = None  # tracked so we can kill the same port later


def find_free_port(start: int = 7860, end: int = 7890) -> int:
    """Pick the first available port."""
    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind(("0.0.0.0", port))
                return port
            except OSError:
                continue
    raise OSError(f"No free port in {start}-{end}. Run stop_gradio() or restart kernel.")


def stop_gradio() -> None:
    """Stop the running Gradio server without restarting the Jupyter kernel."""
    global GRADIO_PORT

    try:
        demo.close()
        print("✓ demo.close()")
    except Exception as exc:
        print(f"demo.close(): {exc}")

    if GRADIO_PORT is not None:
        result = subprocess.run(
            ["fuser", "-k", f"{GRADIO_PORT}/tcp"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            print(f"✓ Freed port {GRADIO_PORT}")
        else:
            print(f"Port {GRADIO_PORT} already free")
        GRADIO_PORT = None


def launch_gradio(port: int | None = 7860) -> int:
    """Stop old server (if any) and launch Gradio. Returns the port used."""
    global GRADIO_PORT

    stop_gradio()
    GRADIO_PORT = port if port is not None else find_free_port()

    print(f"Starting Gradio → http://0.0.0.0:{GRADIO_PORT}")
    demo.launch(
        server_name="0.0.0.0",
        server_port=GRADIO_PORT,
        share=True,
        show_error=True,
        prevent_thread_lock=True,
        theme=gr.themes.Soft(primary_hue="indigo"),
        css=CUSTOM_CSS,
    )
    return GRADIO_PORT


# ── Run this line to start / restart after code changes ──
launch_gradio(7860)   # change to 7861, 7862… if port still stuck

## Section 13 — Example Usage (Programmatic)

Run these cells to test individual agents without the Gradio UI.

In [ ]:
# Example 1: Product search — "recommend running shoes under 3000"
print("=" * 60)
print("EXAMPLE 1: Running shoes under ₹3,000")
print("=" * 60)

response = orchestrator.process(
    message="recommend running shoes under 3000",
    budget=3000,
)
print(response.message)
print(f"\n→ {len(response.recommendations)} recommendations returned")
for rec in response.recommendations:
    p = rec.ranked_product.product
    print(f"  #{rec.rank} {p.title[:60]}... ₹{p.price_inr:.0f} ⭐{p.rating}")

In [ ]:
# Example 2: Complete wedding outfits — budget ₹15,000
print("=" * 60)
print("EXAMPLE 2: Wedding outfits under ₹15,000")
print("=" * 60)

response = orchestrator.process(
    message="I am attending my friend's wedding, budget 15000",
    budget=15000,
    occasion="wedding",
    gender=Gender.FEMALE,
)
print(response.message)
print(f"\n→ {len(response.outfit_recommendations)} outfit boards built")

In [ ]:
# Example 3: Photo-based personalized styling
# Uncomment and provide a photo path to test vision + outfit pipeline:
#
# from pathlib import Path
# photo = Image.open(Path("your_photo.jpg"))
# response = orchestrator.process(
#     message="Generate a wedding outfit for me to wear at my friend's wedding",
#     image=photo,
#     budget=15000,
#     occasion="wedding",
#     gender=Gender.FEMALE,
# )
# print(response.message)
# if response.user_profile:
#     print(f"Profile: {response.user_profile.skin_tone} | {response.user_profile.body_type} | {response.user_profile.style}")

print("Example 3 ready — upload a photo path to test vision-based styling")